In [1]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
import geopandas as gpd
from tqdm import tqdm


In [2]:
ctd = gpd.read_file("../geospatial/ctd_points.geojson")
ctd = ctd.set_crs("EPSG:4326")
ctd.head()

,ControlPointCode,geometry
0,CTD1,POINT (-0.78448 37.8118)
1,CTD2,POINT (-0.8078 37.76062)
2,CTD3,POINT (-0.78355 37.76178)
3,CTD4,POINT (-0.74962 37.74823)
4,CTD5,POINT (-0.72712 37.74045)


In [3]:
raster_path = "../data/Satellite/2021-08-24_strip_4833222_composite.tif"
src = rasterio.open(raster_path)

raster_crs = src.crs
raster_crs

CRS.from_wkt('PROJCS["WGS 84 / UTM zone 30N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-3],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32630"]]')

In [4]:
ctd_proj = ctd.to_crs(raster_crs)
ctd_proj.head()

,ControlPointCode,geometry
0,CTD1,POINT (695024.712 4187246.476)
1,CTD2,POINT (693105.156 4181518.346)
2,CTD3,POINT (695238.476 4181698.092)
3,CTD4,POINT (698264.066 4180265.858)
4,CTD5,POINT (700267.614 4179450.117)


In [5]:
buffer_m = 200
ctd_buffers = ctd_proj.copy()
ctd_buffers["geometry"] = ctd_buffers.buffer(buffer_m)

ctd_buffers.head()

,ControlPointCode,geometry
0,CTD1,"POLYGON ((695224.712 4187246.476, 695223.749 4..."
1,CTD2,"POLYGON ((693305.156 4181518.346, 693304.193 4..."
2,CTD3,"POLYGON ((695438.476 4181698.092, 695437.513 4..."
3,CTD4,"POLYGON ((698464.066 4180265.858, 698463.103 4..."
4,CTD5,"POLYGON ((700467.614 4179450.117, 700466.651 4..."


## Convertir buffers en máscaras de píxeles por CTD

In [6]:
def compute_indices(b):
    coastal = b[0]
    blue = b[1]
    green_i = b[2]
    green = b[3]
    yellow = b[4]
    red = b[5]
    red_edge = b[6]
    nir = b[7]

    ndti = (nir - green) / (nir + green + 1e-6)
    red_green = red / (green + 1e-6)
    blue_green = blue / (green + 1e-6)
    coastal_green = coastal / (green + 1e-6)
    rededge_index = (red_edge - red) / (red_edge + red + 1e-6)

    return np.array([ndti, red_green, blue_green, coastal_green, rededge_index])

In [7]:
ctd = gpd.read_file("../geospatial/ctd_points.geojson").set_crs("EPSG:4326")

In [8]:
image_dir = "../data/Satellite"

tif_files = sorted([
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if f.endswith("composite.tif")
])

CTD + fecha → window → aggregated features → label

In [9]:
def extract_features_from_window(image, mask, nodata=None, min_pixels=50):

    pixels = image[:, mask].T  # (n_pixels, nbands)

    # filter out rows with nodata or NaNs
    if pixels.size == 0:
        return None

    if nodata is not None:
        valid_mask = ~np.any(pixels == nodata, axis=1)
    else:
        valid_mask = ~np.any(np.isnan(pixels), axis=1)

    pixels = pixels[valid_mask]

    if pixels.shape[0] < min_pixels:
        return None

    feats = []

    for band in range(pixels.shape[1]):
        vals = pixels[:, band]

        feats += [
            vals.mean(),
            np.median(vals),
            vals.std(),
            np.percentile(vals, 10),
            np.percentile(vals, 90)
        ]

    # índices por pixel (vectorizado medio)
    coastal = pixels[:, 0]
    blue = pixels[:, 1]
    green_i = pixels[:, 2]
    green = pixels[:, 3]
    yellow = pixels[:, 4]
    red = pixels[:, 5]
    red_edge = pixels[:, 6]
    nir = pixels[:, 7]

    ndti = (nir - green) / (nir + green + 1e-6)
    red_green = red / (green + 1e-6)
    blue_green = blue / (green + 1e-6)
    coastal_green = coastal / (green + 1e-6)
    rededge_index = (red_edge - red) / (red_edge + red + 1e-6)

    for arr in [ndti, red_green, blue_green, coastal_green, rededge_index]:
        feats += [
            np.mean(arr),
            np.std(arr),
            np.median(arr)
        ]

    return np.array(feats)


In [10]:
X_all = []
meta_all = []

In [11]:
for tif_path in tqdm(tif_files, desc="Processing TIFFs"):

    date_match = re.search(r"(\d{4}-\d{2}-\d{2})", os.path.basename(tif_path))
    if not date_match:
        continue

    date = date_match.group(1)

    with rasterio.open(tif_path) as src:

        raster_crs = src.crs
        ctd_proj = ctd.to_crs(raster_crs)

        ctd_buffers = ctd_proj.copy()
        ctd_buffers["geometry"] = ctd_buffers.buffer(buffer_m)

        image = src.read().astype("float32")
        nodata = src.nodata

        for _, row in ctd_buffers.iterrows():

            mask = geometry_mask(
                [row.geometry],
                transform=src.transform,
                invert=True,
                out_shape=(src.height, src.width)
            )

            feat = extract_features_from_window(image, mask, nodata=nodata)

            if feat is None:
                continue

            X_all.append(feat)

            meta_all.append({
                "ctd": row["ControlPointCode"],
                "date": date
            })


In [14]:
# Merge with turbidity labels moved to 04_feature_engineering.ipynb
# df_y loading and merging will be performed there.


In [15]:
meta_df = pd.DataFrame(meta_all)

In [16]:
meta_df["date"] = pd.to_datetime(meta_df["date"]).dt.date
df_y["date"] = pd.to_datetime(df_y["date"]).dt.date
meta_df["ctd"] = meta_df["ctd"].astype(str)
df_y["ctd"] = df_y["ctd"].astype(str)
df = meta_df.merge(df_y, on=["ctd", "date"], how="inner")

In [17]:
df

,ctd,date,turbidity
0,CTD1,2021-08-24,1.542685
1,CTD2,2021-08-24,2.533783
2,CTD3,2021-08-24,1.836980
3,CTD4,2021-08-24,0.997290
4,CTD5,2021-08-24,3.390000
...,...,...,...
462,CTD8,2024-07-03,0.781762
463,CTD9,2024-07-03,0.857323
464,CTD10,2024-07-03,0.871735
465,CTD11,2024-07-03,0.922863


In [18]:
X = np.array(X_all)
y = df["turbidity"].values

In [19]:
X_df = pd.DataFrame(X_all)

In [20]:
save_dir = Path("../data/processed")
save_dir.mkdir(parents=True, exist_ok=True)

X_df.to_parquet(save_dir / "X_features.parquet", index=False)
meta_df.to_parquet(save_dir / "meta.parquet", index=False)
# dataset_final.parquet creation moved to 04_feature_engineering.ipynb
